# M05 · XGBoost

Entrenamos un clasificador **XGBoost** para predecir el grupo NOVA (1 a 4) de un producto.

**Objetivo:** macro F1 lo más alto posible, con un gap entre train y validación **< 5 %**.

## Paso 1 · Carga de datos

Partimos del fichero limpio `off_nova_clean.csv.gz` (lo generó el EDA). **No lo modificamos nunca**: cualquier transformación propia de este modelo irá dentro de un pipeline más adelante.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("data/processed/off_nova_clean.csv.gz")
if not DATA.exists():  # Jupyter lanzado desde notebooks/
    DATA = Path("../data/processed/off_nova_clean.csv.gz")

df = pd.read_csv(DATA, dtype={"code": str})
df.shape

(48119, 35)

## Paso 2 · Separar train y test

La columna `split` ya viene marcada en el fichero (80 % train, 20 % test). La usamos tal cual para que todos los modelos del equipo comparen con los mismos datos.

Separamos **antes de explorar nada**. El test queda guardado y no lo miramos: solo se usa una vez, al final, en M08, para la nota final del modelo elegido.

In [5]:
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

In [6]:
print("train:", train.shape)
print("test:", test.shape)

train: (38495, 35)
test: (9624, 35)


## Paso 3 · Definir X, y y los grupos (solo train)

- `y` es lo que queremos predecir: `nova_group`.
- `X` son las variables con las que el modelo predice. Quitamos columnas que no son features:
  - `code`, `product_name`, `brands`, `ingredients_text`: identificadores o texto libre.
  - `split`: solo sirve para separar.
  - `nova_group`: es la respuesta; si la dejamos en `X`, el modelo hace trampa.
  - `grupo_producto`: lo guardamos aparte en `groups`. No es una feature, pero lo necesita la validación cruzada para que un mismo producto (mismo nombre y marca, varias filas casi iguales) no caiga a la vez en train y en validación. Sin eso, el score saldría inflado.

In [7]:
NOT_FEATURES = ["code", "product_name", "brands", "ingredients_text",
                "split", "nova_group", "grupo_producto"]

y = train["nova_group"]
groups = train["grupo_producto"]
X = train.drop(columns=NOT_FEATURES)

In [8]:
print("X:", X.shape)
print(list(X.columns))
print()
print(y.value_counts().sort_index())
print()
print(y.value_counts(normalize=True).sort_index().round(3))

X: (38495, 28)
['additives_n', 'pnns_groups_1', 'pnns_groups_2', 'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'fruits-vegetables-legumes_100g', 'sin_lista_ingredientes', 'tiene_aromas', 'tiene_edulcorantes', 'tiene_jarabes', 'tiene_almidon_modificado', 'tiene_grasa_hidrogenada', 'tiene_proteina_aislada', 'tiene_colorantes', 'tiene_emulsionantes', 'tiene_potenciadores', 'tiene_aceite_palma', 'tiene_conservantes', 'tiene_sal', 'tiene_azucar', 'tiene_aceite', 'n_ingredientes']

nova_group
1     4952
2     4308
3     7865
4    21370
Name: count, dtype: int64

nova_group
1    0.129
2    0.112
3    0.204
4    0.555
Name: proportion, dtype: float64


### Resultados

- `X` tiene 38.495 filas y 28 columnas: son los productos de train y las variables con las que el modelo va a predecir.
- Hay 4 clases NOVA. La clase 4 tiene 21.370 productos (55,5 %); las otras son mucho más pequeñas: NOVA 3 con 20,4 %, NOVA 1 con 12,9 % y NOVA 2 con 11,2 %.
- Las clases están **desbalanceadas**. Un modelo que dijera siempre "NOVA 4" acertaría más de la mitad de las veces sin aprender nada. Por eso la métrica principal es **macro F1** (que da el mismo peso a cada clase) y no accuracy, y por eso miraremos el recall de cada clase, sobre todo de NOVA 2, que es la más pequeña.